# S6E11 | Three Ladders, Two Steps: What Moved LB to 0.95347

**Our earlier versions** (all in the version history): v16 0.95343, v15 0.95333, v13 0.95330, v12 0.95300, v11 0.95291, v10 0.95288, v9 0.95239.

This version starts from goodpjw2008's [LM Ladder Part 1](https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-part-1-own-lm-step-lb-0-95346) (0.95346), which grew out of our v16 step. It adds the two moves that won our probes today: the **current** megayak language model in place of the frozen 8 November copy, and **half** of our v15 tree move. Its public score is shown in the box above. Most of the score is the work of Moran Z, amanatar, goodpjw2008 and megayak. Read **How to read the gaps** before you use it as a final pick.

## The ladder so far

| Step | By | What changed | Public LB |
|---|---|---|---|
| v12 | us | wide LM stack + shallow-tree fusion, CV-backed | 0.95300 |
| Strike 9 | Moran Z | grandmaster ladder | 0.95330 |
| v15 | us | Strike 9 stepped away from the MLP and GB bags | 0.95333 |
| file | us | + 12% megayak LM | 0.95336 |
| Strike 10 | amanatar | podium apex climb from Strike 9 | 0.95340 |
| v16 | us | Strike 10 + 0.12 (LM - F) | 0.95343 |
| LM Ladder Part 1 | goodpjw2008 | + 0.6 (A - F), the 8 Nov LM, his own seed-7 LM | 0.95346 |
| **v17** | us | **current LM + half tree move** | **0.95347** |

## What it does

1. Loads six public files and checks each against its SHA-256:
   - `submission_095340.csv` (`A`): Strike 10 by amanatar.
   - `submission_095330.csv` (`F`): Strike 9 by Moran Z, the common base of every step.
   - `lm_test.npy` (`LM`): megayak's target-first GPT-2, current version.
   - `lm7_test_f0.npy` (`L7`): goodpjw2008's compact target-first LM (seed 7, fold 0). It holds the log-likelihood of each test row under label False / True for 4 column orders, and the log-odds is `(a[..., 1] - a[..., 0]).mean(1)`.
   - `test_gbn_bag5_w512.npy` (`MLP`) and `test_gb_bag5.npy` (`GB`): from goodpjw2008's OOF bank.
2. Turns each into logit ranks, `logit((rank - 0.5) / n)`. AUC only sees order, so this puts all six on one scale.
3. Builds goodpjw2008's ladder with the current LM, then steps half our tree move away from MLP and GB, all measured from Strike 9:

```
B     = A + 0.6 (A - F) + 0.12 (LM - F) + 0.06 (L7 - F)
score = B - 0.05 (MLP - F) - 0.025 (GB - F)
```

4. Ranks the result, writes `submission.csv` and checks its SHA-256.

Against goodpjw2008's 0.95346 recipe, the only changes are `LM` in place of his frozen copy `ML` and the last line. Its rank correlation with his file is 0.99992.

## Today's probes

Every probe is a one-axis change to goodpjw2008's file `G = A + 0.6 (A - F) + 0.12 (ML - F) + 0.06 (L7 - F)`, which we first rebuilt rank for rank from his inputs. All were file uploads.

| Probe | Public LB |
|---|---|
| G (goodpjw2008, reproduced) | 0.95346 |
| extrapolation 1.0 instead of 0.6 | 0.95346 |
| whole step from A x 1.5 | 0.95342 |
| current megayak LM in place of the 8 Nov copy | 0.95347 |
| G - 0.05 (MLP - F) - 0.025 (GB - F), half our tree move | 0.95347 |
| **both moves (this version)** | **0.95347** |
| both, with the full tree move | 0.95346 |
| both + amanatar's last step (his 0.95339 file to his 0.95340 file) | 0.95346 |
| both + 0.12 (LM - ML), extending the LM version step | 0.95347 |

What the table says:

- The extrapolation is flat from 0.6 to 1.0, and pushing the whole step further loses. goodpjw2008's step sizes are already near the top.
- The newer megayak LM and the half tree move each gain 0.00001, but **they do not add**: together they stay at 0.95347, and four different files share that score.
- The tree move peaks at half size here, while the full move won in v15 from Strike 9. The next section explains why.

## Why half the tree move is enough

In v16 we decomposed amanatar's Strike 10 into moves from Strike 9 (least squares on logit ranks). It carries -0.047 on MLP, -0.034 on GB and almost no language model (-0.001). goodpjw2008's file extrapolates Strike 10 by 1.6, so it carries about -0.075 on MLP and -0.054 on GB, already close to our v15 move (-0.10 / -0.05). Half a move more tops it up, and a full move overshoots. The language-model steps stack because Strike 10 holds almost none.

## CV evidence: these steps do not hold up

Strike 9, Strike 10 and the ladder files have no OOF, so we test our moves on the nested OOF of goodpjw2008's Part 12 wide LM stack (AUC 0.952565), the closest public stand-in:

| Move on the wide-stack OOF | CV change |
|---|---|
| full tree move: 0.10 away from MLP + 0.05 away from GB (v15) | -7.4e-5 |
| + 5% megayak LM (all 5 folds) | -1.5e-5 |
| + 10% megayak LM (fold 0) | -14.7e-5 |

Every move that wins on the public LB loses on CV. We did not CV the half tree move or the LM version swap separately. goodpjw2008's seed-7 LM reaches fold-0 AUC 0.947545 on its own.

## What did not work

- Today: pushing the ladder further (x1.5), the full tree move and amanatar's last-step direction all fell back to 0.95346 or below.
- Earlier: our own small target-first transformer (fold-0 AUC 0.94685 after 45 minutes on a T4) correlated 0.976 with the public LMs and lost CV at every weight. masayakawamata's GPT-2 and more MLP and EBM bags diluted the file (0.95331 to 0.95332 in the v16 probes).

## How to read the gaps

- The public LB is about 20% of the test set, roughly 60k rows. 0.00001 is noise and 0.00003 is barely more. This version is **one point** above 0.95346.
- Strike 9, Strike 10, LM Ladder Part 1 and this notebook all contain LB-gradient steps. All of them can fade on the private 80%.
- **For your two final picks, treat this as the best-public pick and pair it with a CV-backed one**, such as our v12 (0.95300).
- The next real gain is a model with different errors, checked on OOF first, not more blending.

## Credits

- **goodpjw2008**:
  - [S6E11 LM Ladder Part 1 | Own LM Step | LB 0.95346](https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-part-1-own-lm-step-lb-0-95346), the recipe this version starts from.
  - [S6E11 compact target-first LM, seed 7](https://www.kaggle.com/datasets/goodpjw2008/s6e11-compact-lm-seed7), which provides L7.
  - The [S6E11 OOF Bank](https://www.kaggle.com/datasets/goodpjw2008/s6e11-oof-bank), which provides MLP and GB.
  - [Part 12: Wide Stack + 2nd LM](https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298) and [s6e11-fraud-series-artifacts](https://www.kaggle.com/datasets/goodpjw2008/s6e11-fraud-series-artifacts), the OOFs behind every CV check.
- **amanatar**, [S6E11 | Podium Apex | Public LB 0.95340](https://www.kaggle.com/code/amanatar/s6e11-podium-apex-public-lb-0-95340) and the [Strike 10 dataset](https://www.kaggle.com/datasets/amanatar/s6e11-strike10-podium-apex-095340): the 0.95340 file.
- **Moran Z**, [S6E11 | Grandmaster Ladder | LB 0.95330](https://www.kaggle.com/code/moranzavdi/s6e11-grandmaster-ladder-lb-0-95330) and the [Strike9 dataset](https://www.kaggle.com/datasets/moranzavdi/s6e11-strike9-lb-095330): the common base.
- **megayak**:
  - [S6E11 target-first LM predictions](https://www.kaggle.com/datasets/megayak/s6e11-target-first-lm), which provides LM, plus the compact row format and training script behind L7.
  - [S6E11 | Per-Value + Digit TE | Shallow-Tree Stack](https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack), the boosting-from-a-GLM recipe behind GB.
- **Chris Deotte**, [S6E10 1st place writeup](https://www.kaggle.com/competitions/playground-series-s6e10/writeups/1st-place-tfms-and-llms-with-tabular-data): the target-first language-model idea. **masayakawamata**, [GPT-2 Target-First Classifier](https://www.kaggle.com/code/masayakawamata/s6e11-gpt-2-target-first-classifier-cv-0-9500): probed in v16.
- **Malyshev Danil**, [Fraud Signal Lab | GPT-2 and GLM Fusion](https://www.kaggle.com/code/malyshevdanil/fraud-signal-lab-gpt-2-and-glm-fusion): the 65/35 backbone of our earlier versions.
- **alexchilton** ([Shallow-Tree Stack](https://www.kaggle.com/code/alexchilton/credit-card-fraud-lb-0-9524-shallow-tree-stack)): probed in earlier versions, not used.
- The wide stack pools public predictions from Sadam Torres, alexchilton, Malyshev Danil, lazyyeou, megayak, Will, queyrusi, Vladimir Demidov, Nilanjan_Saha123, masayakawamata, Sameer Shaikh and goodpjw2008.

If this helps, please upvote the source notebooks too, starting with goodpjw2008, amanatar, Moran Z and megayak.

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from scipy.special import logit

# Public inputs, checked byte for byte
EXPECTED = {
    'submission_095340.csv': '12b525b84f29e32d6511e9d6105f6dc5eac79ae41b10140eed3353ea57afc159',  # Strike 10 by amanatar
    'submission_095330.csv': 'f4a7a4f8636c99fabd82945c7d81956c5559c609dcb55cde6d4e03f80b0ea105',  # Strike 9 by Moran Z
    'lm_test.npy': '3168779be644a530c6bdc4c141cae168bfc2feb01ac21090c917cc978a7687d3',  # megayak target-first LM, current version
    'lm7_test_f0.npy': '78af7498ee398b300556a8afc9d661fdfd2b42d8d632e6af9cd412aa3430a369',  # goodpjw2008 compact target-first LM, seed 7
    'test_gbn_bag5_w512.npy': '035d6544d7c03108fc42ee296a8660a45825e3b1b19b7cabce1c3f42cac4e7cb',  # goodpjw2008 OOF bank
    'test_gb_bag5.npy': '7d54e5f17e0707d88fec1a2cccb05885978ae6cc2a595c5f4575c7b7d45a9bb8',  # goodpjw2008 OOF bank
}
OUT_SHA = 'ae59376b63430546477c957f1caf1c39670409d346bdcb2936671a18f5082992'  # this version's submission
INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/submission.csv')

def locate(name):
    hits = sorted(INPUT.rglob(name))
    assert hits, name
    return hits[0]

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def logit_rank(v):
    return logit((rankdata(v) - 0.5) / len(v))

paths = {name: locate(name) for name in EXPECTED}
for name, path in paths.items():
    assert sha256(path) == EXPECTED[name], (name, sha256(path))

sample = pd.read_csv(locate('sample_submission.csv'))
strike10 = pd.read_csv(paths['submission_095340.csv'])
strike9 = pd.read_csv(paths['submission_095330.csv'])
for sub in (strike10, strike9):
    assert (sub['id'].to_numpy() == sample['id'].to_numpy()).all()

A = logit_rank(strike10['is_fraud'].to_numpy())
F = logit_rank(strike9['is_fraud'].to_numpy())
LM = logit_rank(np.load(paths['lm_test.npy']))
a7 = np.load(paths['lm7_test_f0.npy'])  # [rows, 4 column orders, log p(row | False / True)]
L7 = logit_rank((a7[..., 1] - a7[..., 0]).mean(1))
MLP = logit_rank(np.load(paths['test_gbn_bag5_w512.npy']))
GB = logit_rank(np.load(paths['test_gb_bag5.npy']))
assert len(A) == len(F) == len(LM) == len(L7) == len(MLP) == len(GB) == len(sample)

# goodpjw2008's LM ladder, with the current megayak LM
B = A + 0.6 * (A - F) + 0.12 * (LM - F) + 0.06 * (L7 - F)
# half of our v15 tree move: a step away from the MLP and GB bags
score = B - 0.05 * (MLP - F) - 0.025 * (GB - F)
print('correlation with Strike 10:', round(float(np.corrcoef(score, A)[0, 1]), 6))

sample.assign(is_fraud=(rankdata(score) - 0.5) / len(score)).to_csv(OUT, index=False)
assert sha256(OUT) == OUT_SHA, sha256(OUT)
print('wrote', OUT, len(sample), 'rows, sha', OUT_SHA[:16])

In [ ]:
import pandas as pd
sub = pd.read_csv('/kaggle/working/submission.csv')
print(sub.shape)
print(sub.head())
print(sub['is_fraud'].describe())